# 第 6 章：逆矩陣與 LU 分解

本 Notebook 是 `ch06_inverse_lu.md` 的精簡對照版本，程式碼邏輯與 `ch06_inverse_lu.py` 一致，可互動執行。

## 學習目標

- 逆矩陣的定義與存在條件（det(A) != 0）
- 2x2 逆矩陣公式，並用高斯-若丹消去法手算 3x3 逆矩陣
- 逆矩陣的性質：(AB)^-1 = B^-1 A^-1、(A^T)^-1 = (A^-1)^T、(A^-1)^-1 = A
- LU 分解 A=LU 的概念與用途


In [1]:
import numpy as np
from scipy.linalg import lu

np.set_printoptions(precision=4, suppress=True)


## 1. 2x2 逆矩陣

公式：A^-1 = (1/det(A)) * [[d, -b], [-c, a]]

範例：A = [[2, 1], [1, 1]]，det(A) = 2(1)-1(1) = 1，所以 A^-1 = [[1, -1], [-1, 2]]。


In [2]:
A2 = np.array([[2.0, 1.0],
               [1.0, 1.0]])
print("矩陣 A =")
print(A2)

det_A2 = np.linalg.det(A2)
print("det(A) =", det_A2)

A2_inv = np.linalg.inv(A2)
print("A 的逆矩陣 A_inv =")
print(A2_inv)

a, b = A2[0, 0], A2[0, 1]
c, d = A2[1, 0], A2[1, 1]
A2_inv_formula = (1.0 / det_A2) * np.array([[d, -b], [-c, a]])
print("用公式手算的逆矩陣 =")
print(A2_inv_formula)
print("公式結果與 numpy 結果是否一致:", np.allclose(A2_inv, A2_inv_formula))

I2 = np.eye(2)
check1 = np.allclose(A2 @ A2_inv, I2)
print("驗證 A @ A_inv ≈ I:", check1)
assert check1


矩陣 A =
[[2. 1.]
 [1. 1.]]
det(A) = 1.0
A 的逆矩陣 A_inv =
[[ 1. -1.]
 [-1.  2.]]
用公式手算的逆矩陣 =
[[ 1. -1.]
 [-1.  2.]]
公式結果與 numpy 結果是否一致: True
驗證 A @ A_inv ≈ I: True


## 2. 3x3 逆矩陣（對照高斯-若丹消去法手算結果）

講義中對 A = [[1,2,3],[0,1,4],[5,6,0]] 用高斯-若丹消去法（[A|I] -> [I|A^-1]）手算，得到：

A^-1 = [[-24, 18, 5], [20, -15, -4], [-5, 4, 1]]

以下用 `numpy.linalg.inv` 驗證這個手算結果。


In [3]:
A3 = np.array([[1.0, 2.0, 3.0],
               [0.0, 1.0, 4.0],
               [5.0, 6.0, 0.0]])
print("矩陣 A =")
print(A3)

A3_inv = np.linalg.inv(A3)
print("numpy 計算出的逆矩陣 A_inv =")
print(A3_inv)

A3_inv_hand = np.array([[-24.0, 18.0, 5.0],
                         [20.0, -15.0, -4.0],
                         [-5.0, 4.0, 1.0]])
print("numpy 結果與講義手算結果是否一致:", np.allclose(A3_inv, A3_inv_hand))

I3 = np.eye(3)
check2 = np.allclose(A3 @ A3_inv, I3)
print("驗證 A @ A_inv ≈ I:", check2)
assert check2


矩陣 A =
[[1. 2. 3.]
 [0. 1. 4.]
 [5. 6. 0.]]
numpy 計算出的逆矩陣 A_inv =
[[-24.  18.   5.]
 [ 20. -15.  -4.]
 [ -5.   4.   1.]]
numpy 結果與講義手算結果是否一致: True
驗證 A @ A_inv ≈ I: True


## 3. 奇異矩陣（singular matrix）無法求逆

若 det(A) = 0，`numpy.linalg.inv` 會拋出 `LinAlgError`。


In [4]:
S = np.array([[2.0, 4.0],
              [1.0, 2.0]])
print("矩陣 S =")
print(S)
print("det(S) =", np.linalg.det(S))

try:
    np.linalg.inv(S)
except np.linalg.LinAlgError as e:
    print("np.linalg.inv 拋出例外，如預期:", e)


矩陣 S =
[[2. 4.]
 [1. 2.]]
det(S) = 0.0
np.linalg.inv 拋出例外，如預期: Singular matrix


## 4. 逆矩陣的性質

- (AB)^-1 = B^-1 A^-1
- (A^T)^-1 = (A^-1)^T
- (A^-1)^-1 = A


In [5]:
A = np.array([[2.0, 1.0],
              [1.0, 1.0]])
B = np.array([[3.0, 0.0],
              [1.0, 2.0]])

A_inv = np.linalg.inv(A)
B_inv = np.linalg.inv(B)

AB_inv = np.linalg.inv(A @ B)
print("性質 1: (AB)^-1 == B^-1 A^-1 ?", np.allclose(AB_inv, B_inv @ A_inv))

AT_inv = np.linalg.inv(A.T)
print("性質 2: (A^T)^-1 == (A^-1)^T ?", np.allclose(AT_inv, A_inv.T))

print("性質 3: (A^-1)^-1 == A ?", np.allclose(np.linalg.inv(A_inv), A))


性質 1: (AB)^-1 == B^-1 A^-1 ? True
性質 2: (A^T)^-1 == (A^-1)^T ? True
性質 3: (A^-1)^-1 == A ? True


## 5. LU 分解：不需 pivoting 的簡單範例

範例：A = [[6, 3], [4, 3]]（第一列主元已是該行絕對值最大的元素，scipy 的 partial pivoting 不會實際換列）。

手算：乘數 m21 = 4/6 = 2/3，R2 <- R2 - (2/3)R1 得 U=[[6,3],[0,1]]，L=[[1,0],[2/3,1]]。

`scipy.linalg.lu` 回傳的慣例是 A = P L U（即 P^T A = LU）。


In [6]:
M = np.array([[6.0, 3.0],
              [4.0, 3.0]])
print("矩陣 M =")
print(M)

P, L, U = lu(M)
print("P =")
print(P)
print("L =")
print(L)
print("U =")
print(U)

check3 = np.allclose(L @ U, P.T @ M)
print("驗證 L @ U ≈ P.T @ M:", check3)
assert check3

print("P 是否為單位矩陣（代表此例不需要 pivoting）:", np.allclose(P, np.eye(2)))

det_M = np.linalg.det(M)
det_via_LU = np.linalg.det(L) * np.linalg.det(U) * np.linalg.det(P)
print("det(M) (直接計算) =", det_M)
print("det(L)*det(U)*det(P) (透過 LU 分解計算) =", det_via_LU)


矩陣 M =
[[6. 3.]
 [4. 3.]]
P =
[[1. 0.]
 [0. 1.]]
L =
[[1.     0.    ]
 [0.6667 1.    ]]
U =
[[6. 3.]
 [0. 1.]]
驗證 L @ U ≈ P.T @ M: True
P 是否為單位矩陣（代表此例不需要 pivoting）: True
det(M) (直接計算) = 6.0
det(L)*det(U)*det(P) (透過 LU 分解計算) = 6.0


## 6. LU 分解：3x3 範例（一般情況，含 permutation）

以及應用：用 LU 分解（透過 `np.linalg.solve`，其內部也是基於 LU 分解）快速求解 Nx=b。


In [7]:
N = np.array([[1.0, 2.0, 3.0],
              [0.0, 1.0, 4.0],
              [5.0, 6.0, 0.0]])
print("矩陣 N =")
print(N)

P2, L2, U2 = lu(N)
print("P =")
print(P2)
print("L =")
print(L2)
print("U =")
print(U2)

check4 = np.allclose(L2 @ U2, P2.T @ N)
print("驗證 L @ U ≈ P.T @ N:", check4)
assert check4

b_vec = np.array([1.0, 2.0, 3.0])
x = np.linalg.solve(N, b_vec)
print("解 Nx = b, 其中 b =", b_vec)
print("解得 x =", x)
print("驗證 N @ x ≈ b:", np.allclose(N @ x, b_vec))


矩陣 N =
[[1. 2. 3.]
 [0. 1. 4.]
 [5. 6. 0.]]
P =
[[0. 0. 1.]
 [0. 1. 0.]
 [1. 0. 0.]]
L =
[[1.  0.  0. ]
 [0.  1.  0. ]
 [0.2 0.8 1. ]]
U =
[[ 5.   6.   0. ]
 [ 0.   1.   4. ]
 [ 0.   0.  -0.2]]
驗證 L @ U ≈ P.T @ N: True
解 Nx = b, 其中 b = [1. 2. 3.]
解得 x = [ 27. -22.   6.]
驗證 N @ x ≈ b: True


## 重點整理

- 逆矩陣 A^-1 滿足 AA^-1=A^-1A=I，存在條件為 det(A) != 0。
- 2x2 逆矩陣可用公式直接計算；更大的矩陣可用高斯-若丹消去法 [A|I] -> [I|A^-1]。
- 逆矩陣性質：(AB)^-1=B^-1A^-1、(A^T)^-1=(A^-1)^T、(A^-1)^-1=A。
- LU 分解 A=LU（或含排列的 PA=LU）可加速多組 Ax=b 求解與行列式計算。
- Python 常用 `numpy.linalg.inv` 與 `scipy.linalg.lu`；注意 scipy 的 `lu` 預設回傳 A=PLU 慣例，且預設採用 partial pivoting。
